In [1]:
from unsloth import FastLanguageModel
import torch
import pandas as pd
import re
from tqdm import tqdm

model_name = "unsloth/Qwen3.5-4B" # Note: Qwen 3.5 4B is the current recommendation
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model) # 2x faster inference

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/rohan/nyu_svg_contest/unsloth_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 24.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 723/723 [00:03<00:00, 184.82it/s]


Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear4bit(in_features=1024, out_features=3072, bias=True)
            (proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear4bit(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear4bit(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
   

In [2]:
import re

def generate_svg(prompt):
    # Strict Qwen 3.5 Vision-Language dictionary format for pure text
    messages = [
        {
            "role": "system", 
            "content": [{"type": "text", "text": "You are an expert SVG developer. Generate ONLY valid, minimal SVG code. Do not include markdown formatting or explanations. The SVG must be width='256' height='256' with viewBox='0 0 256 256'."}]
        },
        {
            "role": "user", 
            "content": [{"type": "text", "text": f"Create a simple SVG icon for: {prompt}"}]
        }
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True 
    ).to("cuda")

    # Safely extract the underlying text tokenizer from the VL Processor
    text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

    # Tell the model exactly when to stop generating
    terminators = [
        text_tokenizer.eos_token_id,
        text_tokenizer.convert_tokens_to_ids("<|im_end|>")
    ]

    outputs = model.generate(
        **inputs,
        max_new_tokens=1000, # Hard cap of ~4000 characters
        use_cache=True,
        temperature=0.1,
        pad_token_id=text_tokenizer.eos_token_id,
        eos_token_id=terminators 
    )
    
    # Slice out the prompt so we only decode the model's response
    input_length = inputs["input_ids"].shape[1]
    decoded = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    
    # Regex to grab just the SVG block
    match = re.search(r"<svg.*?</svg>", decoded, re.DOTALL | re.IGNORECASE)
    
    if match:
        return match.group(0).replace('\n', ' ').strip()
    else:
        # Fallback black square if the model completely fails
        return "<svg xmlns='http://www.w3.org/2000/svg' width='256' height='256' viewBox='0 0 256 256'><rect width='256' height='256' fill='black'/></svg>"

In [3]:
import pandas as pd
from tqdm import tqdm

# 1. Load the data (Just grabbing the first 20 for a quick test)
test_df = pd.read_csv('csv/test.csv')
sample_test_df = test_df.head(20) 

results = []

print(f"Generating SVGs for {len(sample_test_df)} prompts...")

# 2. Run the inference loop
for index, row in tqdm(sample_test_df.iterrows(), total=len(sample_test_df)):
    svg_output = generate_svg(row['prompt'])
    
    # Store results in a standard Python list
    results.append({
        'id': row['id'],
        'svg': svg_output
    })

# 3. Save everything to CSV at the very end
output_file = 'csv/baseline_sample_clean.csv'
submission_df = pd.DataFrame(results)
submission_df.to_csv(output_file, index=False)

print(f"\nDone! Results saved to {output_file}")

Generating SVGs for 20 prompts...


100%|██████████| 20/20 [09:42<00:00, 29.12s/it]


Done! Results saved to csv/baseline_sample_clean.csv


In [4]:
import pandas as pd
import re
import torch
from tqdm import tqdm

# 1. Prepare the Tokenizer for Batching
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
text_tokenizer.padding_side = 'left' # Critical for batch generation
if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token

def generate_svg_batch(prompts):
    # Format the entire batch of prompts
    messages_batch = [
        [
            {"role": "system", "content": [{"type": "text", "text": "You are an expert SVG developer. Generate ONLY valid, minimal SVG code. Do not include markdown formatting or explanations. The SVG must be width='256' height='256' with viewBox='0 0 256 256'."}]},
            {"role": "user", "content": [{"type": "text", "text": f"Create a simple SVG icon for: {prompt}"}]}
        ]
        for prompt in prompts
    ]
    
    # Apply chat template to turn dictionaries into raw string prompts
    texts = [text_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) for msgs in messages_batch]
    
    # Tokenize the entire batch at once
    inputs = text_tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to("cuda")

    terminators = [
        text_tokenizer.eos_token_id,
        text_tokenizer.convert_tokens_to_ids("<|im_end|>")
    ]

    # Generate outputs for the whole batch
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000, 
        use_cache=True,
        temperature=0.1,
        pad_token_id=text_tokenizer.pad_token_id,
        eos_token_id=terminators 
    )
    
    # Decode the batch (slicing off the prompt lengths)
    input_length = inputs["input_ids"].shape[1]
    decoded_batch = text_tokenizer.batch_decode(outputs[:, input_length:], skip_special_tokens=True)
    
    # Extract SVGs using Regex
    svgs = []
    for decoded in decoded_batch:
        match = re.search(r"<svg.*?</svg>", decoded, re.DOTALL | re.IGNORECASE)
        if match:
            svgs.append(match.group(0).replace('\n', ' ').strip())
        else:
            svgs.append("<svg xmlns='http://www.w3.org/2000/svg' width='256' height='256' viewBox='0 0 256 256'><rect width='256' height='256' fill='black'/></svg>")
            
    return svgs

# 2. Execution Setup
test_df = pd.read_csv('csv/test.csv') # Loading all 1000 rows
results = []

# --- THE MAGIC NUMBER ---
# Your RTX 3090 has 24GB of VRAM. A batch size of 8 or 16 should be extremely fast. 
# If it gives an "Out of Memory" (OOM) error, lower this to 4.
BATCH_SIZE = 8 

print(f"Generating SVGs for {len(test_df)} prompts with a batch size of {BATCH_SIZE}...")

# 3. The Batch Loop
for i in tqdm(range(0, len(test_df), BATCH_SIZE)):
    # Grab a chunk of rows
    batch_df = test_df.iloc[i:i + BATCH_SIZE]
    prompts = batch_df['prompt'].tolist()
    ids = batch_df['id'].tolist()
    
    # Send the chunk to the GPU
    batch_svgs = generate_svg_batch(prompts)
    
    # Save the results
    for p_id, svg in zip(ids, batch_svgs):
        results.append({
            'id': p_id,
            'svg': svg
        })

# 4. Save Final CSV
output_file = 'csv/baseline_submission_2.csv'
submission_df = pd.DataFrame(results)
submission_df.to_csv(output_file, index=False)

print(f"\nDone! Full baseline saved to {output_file}")

Generating SVGs for 1000 prompts with a batch size of 8...


100%|██████████| 125/125 [3:10:33<00:00, 91.47s/it]  


Done! Full baseline saved to csv/baseline_submission_2.csv


In [8]:
import csv
import re
import os

allowed_elements = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line", "polyline", 
    "polygon", "defs", "use", "symbol", "clipPath", "mask", "linearGradient", 
    "radialGradient", "stop", "text", "tspan", "title", "desc", "style", 
    "pattern", "marker", "filter"
}

def make_kaggle_safe_svg(svg_content):
    svg_content = str(svg_content)
    
    # 1. Strip XML comments and forbidden tags
    svg_content = re.sub(r'', '', svg_content, flags=re.DOTALL)
    def tag_replacer(match):
        tag_full = match.group(0)
        tag_name_match = re.match(r'</?([a-zA-Z0-9]+)', tag_full)
        if tag_name_match and tag_name_match.group(1) in allowed_elements:
            return tag_full
        return ""
    cleaned = re.sub(r'<[^>]+>', tag_replacer, svg_content)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    
    # 2. Add xmlns if missing
    if 'xmlns=' not in cleaned.lower():
        cleaned = re.sub(r'<svg', '<svg xmlns="http://www.w3.org/2000/svg"', cleaned, count=1, flags=re.IGNORECASE)
        
    # --- THE KAGGLE PARSER FIX ---
    # Convert any double quotes inside the SVG to single quotes (valid XML)
    cleaned = cleaned.replace('"', "'")
    
    # Remove all commas (SVG coordinates don't need them, spaces work just as well)
    cleaned = cleaned.replace(',', ' ')
    
    return cleaned

# Use your raw baseline file
input_file = 'csv/baseline_submission_2.csv'
output_file = 'csv/final_submission_perfect.csv'
os.makedirs('csv', exist_ok=True)

print("Formatting SVGs for strict Kaggle compliance...")

with open(input_file, 'r', encoding='utf-8') as fin, open(output_file, 'w', encoding='utf-8') as fout:
    # Use standard CSV reader to properly unpack the original file so we don't duplicate quotes
    reader = csv.reader(fin)
    header = next(reader)
    
    # Write manual header
    fout.write("id,svg\n") 
    
    for row in reader:
        if len(row) == 2:
            svg_id = row[0]
            # row[1] is properly un-escaped now by the csv reader!
            cleaned_svg = make_kaggle_safe_svg(row[1])
            
            # Write raw text directly to the file. NO wrapping quotes.
            fout.write(f"{svg_id},{cleaned_svg}\n")

print(f"Done! Safe file saved to {output_file}. Ready to submit!")

Formatting SVGs for strict Kaggle compliance...
Done! Safe file saved to csv/final_submission_perfect.csv. Ready to submit!
